# 基于 Milvus + LangChain 的对话记忆（使用 DeepSeek）

本 Notebook 演示如何使用 VectorStoreRetrieverMemory 存储对话历史，并基于 Milvus 进行检索。

全部 OpenAI 调用已替换为 DeepSeek API。

## 导入所需模块

In [1]:
import os
CUSTOM_CACHE = r'F:\Teewon\Milvue\models'

os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_classic.memory import VectorStoreRetrieverMemory
from langchain_classic.chains import ConversationChain
from langchain_core.prompts import PromptTemplate

## 配置 DeepSeek API 密钥和端点

In [3]:
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("DEEPSEEK_API_KEY")
os.environ["OPENAI_API_BASE"] = "https://api.deepseek.com/v1"

## 使用 MilvusClient 连接 Docker Milvus

In [4]:
from pymilvus import MilvusClient
mc=MilvusClient(
    uri="http://localhost:19530"
)
print(mc.get_server_version())

COLLECTION_NAME = "LangChainCollection"
has=mc.has_collection(COLLECTION_NAME)
if has:
    mc.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection {COLLECTION_NAME}")

3.0.0


## 初始化向量存储（使用 LangChain 的 Milvus 封装）

In [6]:
from langchain_milvus import Milvus
from langchain_community.embeddings import HuggingFaceEmbeddings
# 使用 OpenAI 兼容的 embeddings（DeepSeek 也支持）
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

# 直接创建 Milvus 向量存储实例，指定 collection 名称和连接 URI
vectordb=Milvus(
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
    connection_args={"uri": "http://localhost:19530"},
    text_field="text",
    vector_field="vector",
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 创建 retriever 和 memory

In [7]:
# 创建 retriever 和 memory
retriever=vectordb.as_retriever(search_kwargs=dict(topk=3))
memory=VectorStoreRetrieverMemory(retriever=retriever)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_25620\2631633847.py:3: LangChainDeprecationWarning: The class `VectorStoreRetrieverMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory=VectorStoreRetrieverMemory(retriever=retriever)


## 存入一些示例对话（记忆）

In [8]:
about_me = [
    {"input": "My favorite snack is chocolate", "output": "Nice"},
    {"input": "My favorite sport is swimming", "output": "Cool"},
    {"input": "My favorite beer is Guinness", "output": "Great"},
    {"input": "My favorite dessert is cheesecake", "output": "Good to know"},
    {"input": "My favorite musician is Taylor Swift", "output": "Same"}
]

for example in about_me:
    memory.save_context({"input": example["input"]}, {"output": example["output"]})

F:\Teewon\Milvue\.venv\Lib\site-packages\langchain_milvus\vectorstores\milvus.py:1408: UserWarning: No ids provided and auto_id is False. Setting auto_id to True automatically.
  warnings.warn(


## 测试 memory 是否能检索到相关记忆

In [9]:
print(memory.load_memory_variables({"prompt":"who is my favorite musician?"})["history"])

input: My favorite musician is Taylor Swift
output: Same
input: My favorite sport is swimming
output: Cool
input: My favorite snack is chocolate
output: Nice
input: My favorite beer is Guinness
output: Great


## 定义 Prompt 模板和对话链

In [10]:
_DEFAULT_TEMPLATE = """The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Relevant pieces of previous conversation:
{history}

(You do not need to use these pieces of information if not relevant)

Current conversation:
Human: {input}
AI:"""

In [13]:
PROMPT = PromptTemplate(
    input_variables=["history", "input"], template=_DEFAULT_TEMPLATE
)
llm=ChatOpenAI(
    model="deepseek-v4-flash",
    temperature=0
)

conversation=ConversationChain(
    llm=llm,
    prompt=PROMPT,
    memory=memory,
    verbose=True,
)

## 进行对话测试

In [14]:
response1 = conversation.predict(input="Hi, my name is Gary, what's up?")
print("AI:", response1)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Relevant pieces of previous conversation:
input: My favorite beer is Guinness
output: Great
input: My favorite snack is chocolate
output: Nice
input: My favorite dessert is cheesecake
output: Good to know
input: My favorite sport is swimming
output: Cool

(You do not need to use these pieces of information if not relevant)

Current conversation:
Human: Hi, my name is Gary, what's up?
AI:

> Finished chain.
AI: Hey Gary! Not much, just here and ready to chat. What’s on your mind today?


In [15]:
response2 = conversation.predict(input="who is my favorite musician?")
print("AI:", response2)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Relevant pieces of previous conversation:
input: My favorite musician is Taylor Swift
output: Same
input: My favorite sport is swimming
output: Cool
input: My favorite snack is chocolate
output: Nice
input: Hi, my name is Gary, what's up?
response: Hey Gary! Not much, just here and ready to chat. What’s on your mind today?

(You do not need to use these pieces of information if not relevant)

Current conversation:
Human: who is my favorite musician?
AI:

> Finished chain.
AI: Your favorite musician is Taylor Swift!


In [16]:
response3 = conversation.predict(input="Whats my favorite dessert?")
print("AI:", response3)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Relevant pieces of previous conversation:
input: My favorite dessert is cheesecake
output: Good to know
input: My favorite snack is chocolate
output: Nice
input: My favorite sport is swimming
output: Cool
input: My favorite beer is Guinness
output: Great

(You do not need to use these pieces of information if not relevant)

Current conversation:
Human: Whats my favorite dessert?
AI:

> Finished chain.
AI: Your favorite dessert is cheesecake! 🍰


In [17]:
response4 = conversation.predict(input="What's my name?")
print("AI:", response4)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Relevant pieces of previous conversation:
input: Hi, my name is Gary, what's up?
response: Hey Gary! Not much, just here and ready to chat. What’s on your mind today?
input: My favorite musician is Taylor Swift
output: Same
input: who is my favorite musician?
response: Your favorite musician is Taylor Swift!
input: My favorite snack is chocolate
output: Nice

(You do not need to use these pieces of information if not relevant)

Current conversation:
Human: What's my name?
AI:

> Finished chain.
AI: Your name is Gary!


## 清理资源（可选）

In [ ]:
mc.close()